## Описание/Пошаговая инструкция выполнения домашнего задания:

В качестве данных выберете возьмите датасет RuCoLA для русского языка https://github.com/RussianNLP/RuCoLA (в качестве train возьмите in_domain_train.csv, а в качестве теста in_domain_dev.csv).

Разбейте in_domain_train на train и val.

Зафайнтьюньте и протестируйте RuBert или RuRoBerta на данной задаче (можно взять любую предобученную модель руберт с сайта huggingface. Например, ruBert-base/large https://huggingface.co/sberbank-ai/ruBert-base / https://huggingface.co/sberbank-ai/ruBert-large или rubert-base-cased https://huggingface.co/DeepPavlov/rubert-base-cased, ruRoberta-large https://huggingface.co/sberbank-ai/ruRoberta-large, xlm-roberta-base https://huggingface.co/xlm-roberta-base).


Возьмите RuGPT3 base или large и решите данное задание с помощью методов few-/zero-shot.

а) переберите несколько вариантов затравок;

б) протестируйте различное число few-shot примеров (0, 1, 2, 4).

Обучите и протестируйте модель RuT5 на данной задаче (пример finetun’а можете найти здесь https://github.com/RussianNLP/RuCoLA/blob/main/baselines/finetune_t5.py).

Сравните полученные результаты.


Критерии оценки:

Файнтьюн БЕРТА — 3 балла;

Реализация few-/zero-shot с GPT-3 — 4 балла;

Файнтьюн RuT-5 — 3 балла.

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split

import numpy as np

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import DataCollatorWithPadding
from datasets import Dataset

import warnings
warnings.filterwarnings("ignore")

In [3]:
train_url = 'https://raw.githubusercontent.com/RussianNLP/RuCoLA/main/data/in_domain_train.csv'
dev_url = 'https://raw.githubusercontent.com/RussianNLP/RuCoLA/main/data/in_domain_dev.csv'

train_df = pd.read_csv(train_url)
test_df = pd.read_csv(dev_url)

In [4]:
train_df.head()

,id,sentence,acceptable,error_type,detailed_source
0,0,"Вдруг решетка беззвучно поехала в сторону, и н...",1,0,Paducheva2004
1,1,Этим летом не никуда ездили.,0,Syntax,Rusgram
2,2,Только Иван выразил какую бы то ни было готовн...,1,0,Paducheva2013
3,3,"Теперь ты видишь собственными глазами, как тут...",1,0,Paducheva2010
4,4,На поверку вся теория оказалась полной чепухой.,1,0,Paducheva2010


In [5]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7869 entries, 0 to 7868
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   id               7869 non-null   int64 
 1   sentence         7869 non-null   object
 2   acceptable       7869 non-null   int64 
 3   error_type       7869 non-null   object
 4   detailed_source  7869 non-null   object
dtypes: int64(2), object(3)
memory usage: 307.5+ KB


In [6]:
test_df.head()

,id,sentence,acceptable,error_type,detailed_source
0,0,Иван вчера не позвонил.,1,0,Paducheva2013
1,1,"У многих туристов, кто посещают Кемер весной, ...",0,Syntax,USE8
2,2,Лесные запахи набегали волнами; в них смешалос...,1,0,USE5
3,3,Вчера президент имел неофициальную беседу с ан...,1,0,Seliverstova
4,4,Коллега так и не признал вину за катастрофу пе...,1,0,Testelets


In [7]:
test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 983 entries, 0 to 982
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   id               983 non-null    int64 
 1   sentence         983 non-null    object
 2   acceptable       983 non-null    int64 
 3   error_type       983 non-null    object
 4   detailed_source  983 non-null    object
dtypes: int64(2), object(3)
memory usage: 38.5+ KB


In [8]:
train_df, val_df = train_test_split(train_df, test_size=0.2, stratify=train_df['acceptable'], random_state=42)

In [9]:
from datasets import Dataset
train_dataset = Dataset.from_pandas(train_df[['sentence', 'acceptable']])
val_dataset   = Dataset.from_pandas(val_df[['sentence', 'acceptable']])
test_dataset  = Dataset.from_pandas(test_df[['sentence', 'acceptable']])

In [10]:
train_dataset

Dataset({
    features: ['sentence', 'acceptable', '__index_level_0__'],
    num_rows: 6295
})

In [11]:
val_dataset

Dataset({
    features: ['sentence', 'acceptable', '__index_level_0__'],
    num_rows: 1574
})

In [12]:
test_dataset

Dataset({
    features: ['sentence', 'acceptable'],
    num_rows: 983
})

In [13]:
train_dataset = train_dataset.remove_columns(['__index_level_0__'])
val_dataset = val_dataset.remove_columns(['__index_level_0__'])

## RuBert

In [14]:
model_name = "ai-forever/ruBert-base"
tokenizer_rubert = AutoTokenizer.from_pretrained(model_name)
model_rubert = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ai-forever/ruBert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initia

In [15]:
def tokenize_dataset(dataset):
        return tokenizer_rubert(
            dataset["sentence"],
            truncation=True,
            padding="max_length",
            max_length=128,
        )

In [16]:
train_ds = train_dataset.map(tokenize_dataset, batched=True)
val_ds = val_dataset.map(tokenize_dataset, batched=True)
test_ds = test_dataset.map(tokenize_dataset, batched=True)

Map:   0%|          | 0/6295 [00:00<?, ? examples/s]

Map:   0%|          | 0/1574 [00:00<?, ? examples/s]

Map:   0%|          | 0/983 [00:00<?, ? examples/s]

In [17]:
train_ds = train_ds.rename_column("acceptable", "labels")
val_ds   = val_ds.rename_column("acceptable", "labels")
test_ds  = test_ds.rename_column("acceptable", "labels")

In [18]:
train_ds

Dataset({
    features: ['sentence', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 6295
})

In [19]:
val_ds

Dataset({
    features: ['sentence', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 1574
})

In [20]:
test_ds

Dataset({
    features: ['sentence', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 983
})

In [21]:
from transformers import TrainingArguments
from transformers import Trainer
import evaluate

metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return metric.compute(predictions=preds, references=labels)

training_args = TrainingArguments(
    output_dir="rubert_base",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=10,
    eval_strategy="epoch",              
    save_strategy="epoch",
    load_best_model_at_end=True,
    weight_decay=0.01,
    report_to="none",
    metric_for_best_model="eval_accuracy",
)

trainer = Trainer(
    model=model_rubert,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics = compute_metrics
)

In [22]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.532155,0.494390,0.789708
2,0.357038,0.574993,0.794155
3,0.222787,0.876702,0.796061
4,0.128229,1.008076,0.799238
5,0.118810,1.317650,0.805591
6,0.058298,1.455737,0.796061
7,0.034656,1.567755,0.807497
8,0.024796,1.698890,0.804956
9,0.013083,1.720989,0.807497
10,0.017868,1.721673,0.806861


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (3 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=7870, training_loss=0.1466281403125923, metrics={'train_runtime': 756.5357, 'train_samples_per_second': 83.208, 'train_steps_per_second': 10.403, 'total_flos': 4140710233728000.0, 'train_loss': 0.1466281403125923, 'epoch': 10.0})

In [23]:
result = trainer.predict(test_dataset=test_ds)
print(result.metrics)

{'test_loss': 1.5464468002319336, 'test_accuracy': 0.8056968463886063, 'test_runtime': 2.8448, 'test_samples_per_second': 345.543, 'test_steps_per_second': 43.237}


## RuT5

In [24]:
from transformers import (
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    T5Tokenizer,
    T5ForConditionalGeneration,
)

model_name = "ai-forever/ruT5-base"
tokenizer_rutfive = T5Tokenizer.from_pretrained(model_name)
model_rutfive = T5ForConditionalGeneration.from_pretrained(model_name)

Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [25]:
import evaluate

POS_LABEL = "yes"
NEG_LABEL = "no"

metric = evaluate.load("accuracy")

def compute_metrics(p, tokenizer):
    string_preds = tokenizer.batch_decode(p.predictions, skip_special_tokens=True)
    int_preds = [1 if prediction == POS_LABEL else 0 for prediction in string_preds]

    labels = np.where(p.label_ids != -100, p.label_ids, tokenizer.pad_token_id)
    string_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    int_labels = []

    for string_label in string_labels:
        if string_label == POS_LABEL:
            int_labels.append(1)
        elif string_label == NEG_LABEL or string_label == "":  # second case accounts for test data
            int_labels.append(0)
        else:
            raise ValueError()

    result = metric.compute(predictions=int_preds, references=int_labels)

    return result

In [26]:
def tokenize_dataset(dataset, tokenizer=tokenizer_rutfive):
    result = tokenizer(dataset["sentence"], padding=False)

    if "acceptable" in dataset:
        label_sequences = []
        for label in dataset["acceptable"]:
            if label == 1:
                target_sequence = POS_LABEL
            elif label == 0:
                target_sequence = NEG_LABEL
            else:
                raise ValueError("Unknown class label")
            label_sequences.append(target_sequence)

    else:
        # a hack to avoid the "You have to specify either decoder_input_ids or decoder_inputs_embeds" error
        # for test data
        label_sequences = ["" for _ in dataset["sentence"]]

    result["labels"] = tokenizer(label_sequences, padding=False)["input_ids"]
 
    return result


In [27]:
train_ds = train_dataset.map(tokenize_dataset, batched=True)
val_ds = val_dataset.map(tokenize_dataset, batched=True)
test_ds = test_dataset.map(tokenize_dataset, batched=True)

Map:   0%|          | 0/6295 [00:00<?, ? examples/s]

Map:   0%|          | 0/1574 [00:00<?, ? examples/s]

Map:   0%|          | 0/983 [00:00<?, ? examples/s]

In [28]:
training_args = Seq2SeqTrainingArguments(
    output_dir="rutfive_base",
    eval_strategy="epoch",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=1e-5,
    weight_decay=0.0,
    num_train_epochs=10,
    save_strategy="epoch",
    save_total_limit=1,
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="eval_accuracy",
    predict_with_generate=True,
)


In [29]:
from functools import partial

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer_rutfive, pad_to_multiple_of=8)

trainer = Seq2SeqTrainer(
    model=model_rutfive,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=partial(compute_metrics, tokenizer=tokenizer_rutfive),
    data_collator=data_collator,
)

In [30]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,1.199555,0.205381,0.750318
2,0.235046,0.197277,0.750953
3,0.230497,0.218048,0.702668
4,0.211900,0.203989,0.760483
5,0.204095,0.199744,0.761118
6,0.194313,0.210125,0.761753
7,0.186840,0.214925,0.763659
8,0.188246,0.235243,0.761753
9,0.179990,0.218762,0.765565
10,0.180087,0.228689,0.764295


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (2 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=7870, training_loss=0.26833145839589373, metrics={'train_runtime': 1790.2828, 'train_samples_per_second': 35.162, 'train_steps_per_second': 4.396, 'total_flos': 2526412251856896.0, 'train_loss': 0.26833145839589373, 'epoch': 10.0})

In [31]:
result = trainer.predict(test_dataset=test_ds)
print(result.metrics)

{'test_loss': 0.21289584040641785, 'test_accuracy': 0.7873855544252288, 'test_runtime': 17.8776, 'test_samples_per_second': 54.985, 'test_steps_per_second': 6.88}


## GPT-3


In [32]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "ai-forever/rugpt3large_based_on_gpt2"
tokenizer_gpt = AutoTokenizer.from_pretrained(model_name)
model_gpt = AutoModelForCausalLM.from_pretrained(model_name)

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: ai-forever/rugpt3large_based_on_gpt2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...23}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [33]:
from transformers import pipeline
generator = pipeline("text-generation", 
                     model=model_gpt, 
                     tokenizer=tokenizer_gpt,    
                     do_sample=False, 
                     max_new_tokens=2,
                     temperature=0,
                     max_length=None
                     )

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'max_length', 'do_sample', 'temperature', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [34]:
prompts = [ 
    "Содержит ли текст ошибки? 'да' или 'нет'?\n",

    "Тебе будет дан текст. Проверь, есть ли в нем какие-либо ошибки. Дай ответ: да или нет.\n",

    "Ты работаешь корректором текстов. Тебе дадут текст содержащий разного рода ошибки (орфографические, грамматические, пунктуационные). "
    "Найди эти ошибки и дай ответ одним словом: 'да' - если текст содержит ошибки и 'нет' - если ошибки отсутствуют.\n",
]

In [35]:
def create_question(prompt, text, examples=None):
    question = prompt
    if examples:
        question+= "Пример текста с ответами:\n"
        for sentence, answer in examples:
            if answer == 1:
                ans = 'да'
            else:
                ans = 'нет'  
            question+= f" Текст: {sentence} Твой ответ: {ans}.\n"
    question += f" Текст: {text} Твой ответ:"
    return question

In [36]:
def extract_answer(text):
    answer = text.split(' Твой ответ: ')[-1].strip().lower()
    if "да" in answer and "нет" not in answer:
        return 1
    return 0

In [37]:
from tqdm import tqdm
import evaluate
metric = evaluate.load("accuracy")

def shot_evaluate(train_df, test_df, prompts, shots):
    for i, prompt in enumerate(prompts):
        for n in shots:
            preds = []
            for row in tqdm(test_df.itertuples(), total=len(test_df), leave=False):
                examples = None
                if n: 
                    examples = list(train_df.sample(n, random_state=42)[['sentence', 'acceptable']].itertuples(index=False, name=None))
                question = create_question(prompt, row.sentence, examples)
                gen_answer = generator(question)[0]['generated_text']
                pred = extract_answer(gen_answer)
                preds.append(pred)
            print(f"Prompt {i} - {n} shots - Accuracy:", metric.compute(predictions=preds,references=test_df['acceptable'])['accuracy'])

In [38]:
shots = [0, 1, 2, 4]
shot_evaluate(train_df, test_df, prompts, shots)

  1%|          | 9/983 [00:00<01:04, 15.13it/s]You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Prompt 0 - 0 shots - Accuracy: 0.35096642929806715


Prompt 0 - 1 shots - Accuracy: 0.2777212614445575


Prompt 0 - 2 shots - Accuracy: 0.28687690742624616


Prompt 0 - 4 shots - Accuracy: 0.40793489318413023


Prompt 1 - 0 shots - Accuracy: 0.6653102746693794


... (3 outputs left out)

Prompt 2 - 0 shots - Accuracy: 0.7080366225839267


Prompt 2 - 1 shots - Accuracy: 0.30010172939979657


Prompt 2 - 2 shots - Accuracy: 0.2767039674465921


Prompt 2 - 4 shots - Accuracy: 0.41810783316378436


## Полученные результаты

| Модель | Accuracy | 
| ----------- | ----------- | 
| RuBert    | 0.8056968463886063 | 
| ruT5     | 0.7873855544252288 | 
| GPT-3    | 0.7080366225839267 | 